# Setup


## Параметры

Настраиваемые параметры и пути пайплайна налогового риска.
Все ключевые допущения собраны здесь — см. `docs/adr/`.


In [ ]:
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["font.size"] = 12


In [ ]:
# --- данные ---
CONFIG = {
    "data_path": "test_dataset.csv",
    "data_path_fallback": "test_dataset_smp.csv",
    "out_dir": "output",
    # --- бенчмаркинг (ADR-004) ---
    "group_col": "okved_section",       # отрасль для сравнения с аналогами
    "min_cell_size": 8,                 # минимум компаний в отрасли, иначе — глобальная норма
    "robust_z_flag": 3.5,               # порог |модифицированного z| для флага
    "identity_tol": 0.005,              # допуск для проверки тождеств P&L (0,5%)
    # --- типологии ---
    "shell_headcount_max": 5,           # потолок численности для «транзитной» фирмы
    "young_age_max": 2,                 # «молодая» = возраст (year - ogrn_year) <= 2
    "yoy_flag_z": 3.5,                  # порог z для скачка выручки YoY
    # --- скоринг (ADR-005) ---
    "tier_weights": {"1": 2, "2": 1},  # ведущие типологии весят вдвое больше
    "severity_cap": 8,                  # потолок вклада одной метрики в скор
    # --- корзины (ADR-005) ---
    "likelihood_hi_q": 0.80,
    "materiality_hi_q": 0.80,
    "seed": 42,
}

TIER1_FLAGS = ["flag_low_etr", "flag_shell", "flag_dividend", "flag_margin"]
TIER2_FLAGS = ["flag_identity", "flag_yoy", "flag_young_big", "flag_wage"]
FLAG_SEV_MAP = {
    "flag_low_etr": "sev_low_etr",
    "flag_shell": "sev_shell",
    "flag_dividend": "sev_dividend",
    "flag_margin": "sev_margin",
    "flag_identity": "sev_identity",
    "flag_yoy": "sev_yoy",
    "flag_young_big": "sev_young_big",
    "flag_wage": "sev_wage",
}

Path(CONFIG["out_dir"]).mkdir(parents=True, exist_ok=True)
np.random.seed(CONFIG["seed"])


## Детекторы типологий (ADR-003)

Непрерывные метрики — робастный z-score относительно отрасли с откатом
на глобальную норму для малых ячеек (ADR-004). Арифметические
невозможности — абсолютные булевы правила.


In [ ]:
def _mad_scaled(x):
    """Масштабированное MAD (эквивалент R stats::mad)."""
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan
    med = np.median(x)
    return np.median(np.abs(x - med)) * 1.4826


def _robust_spread(x):
    """Робастный разброс: MAD, при вырождении — IQR / 1.349."""
    spread = _mad_scaled(x)
    if not np.isfinite(spread) or spread == 0:
        x = np.asarray(x, dtype=float)
        x = x[np.isfinite(x)]
        if len(x) == 0:
            return np.nan
        spread = (np.percentile(x, 75) - np.percentile(x, 25)) / 1.349
    return spread if np.isfinite(spread) and spread > 0 else np.nan


def modified_z(x, center, spread):
    """Модифицированный z-score = 0.6745 * (x - median) / MAD."""
    z = 0.6745 * (np.asarray(x, dtype=float) - center) / spread
    return np.where(np.isfinite(z), z, np.nan)


def robust_z_by_group(x, group, min_n=None):
    """Робастный z внутри группы; при малой ячейке — глобальная норма."""
    if min_n is None:
        min_n = CONFIG["min_cell_size"]
    x = np.asarray(x, dtype=float)
    group = np.asarray(group)
    glob_center = np.nanmedian(x)
    glob_spread = _robust_spread(x)
    z = np.full(len(x), np.nan)
    level = np.full(len(x), "global", dtype=object)
    for g in np.unique(group):
        idx = np.where(group == g)[0]
        xv = x[idx]
        n = np.sum(np.isfinite(xv))
        center = np.nanmedian(xv)
        spread = _mad_scaled(xv[np.isfinite(xv)]) if n else np.nan
        if n >= min_n and np.isfinite(spread) and spread > 0:
            z[idx] = modified_z(xv, center, spread)
            level[idx] = "sector"
        else:
            z[idx] = modified_z(xv, glob_center, glob_spread)
            level[idx] = "global"
    return z, level


def _mag(v):
    """Величина z (модуль), NA -> 0, с потолком severity_cap."""
    v = np.abs(np.asarray(v, dtype=float))
    v = np.where(np.isfinite(v), v, 0)
    return np.minimum(v, CONFIG["severity_cap"])


def _lowmag(v):
    """Величина отрицательного хвоста z."""
    v = np.abs(np.minimum(np.asarray(v, dtype=float), 0))
    v = np.where(np.isfinite(v), v, 0)
    return np.minimum(v, CONFIG["severity_cap"])


def _false_na(v):
    """Булев массив: NA трактуется как False."""
    return np.where(pd.isna(np.asarray(v, dtype=bool)), False, np.asarray(v, dtype=bool))


def run_typologies(cy, config=CONFIG):
    """Восемь детекторов типологий: флаг + severity на уровне company-year."""
    g = cy[config["group_col"]].values
    thr = config["robust_z_flag"]

    z_etr, _ = robust_z_by_group(cy["etr"].values, g)
    z_taxrev, _ = robust_z_by_group(cy["tax_to_revenue"].values, g)
    z_rph, _ = robust_z_by_group(cy["revenue_per_head"].values, g)
    z_opm, _ = robust_z_by_group(cy["operating_margin"].values, g)
    z_gm, _ = robust_z_by_group(cy["gross_margin"].values, g)
    z_div, _ = robust_z_by_group(cy["dividend_payout"].values, g)
    z_pph, _ = robust_z_by_group(cy["payroll_per_head"].values, g)
    z_yoy, _ = robust_z_by_group(cy["max_abs_revenue_yoy"].values, g)
    z_rev, _ = robust_z_by_group(cy["revenue"].values, g)

    out = cy[["company_id", "company_name", "year", config["group_col"]]].copy()

    # ---- Tier 1 ----
    # 1. Низкая эффективная ставка налога
    out["flag_low_etr"] = _false_na((z_etr <= -thr) | (z_taxrev <= -thr))
    out["sev_low_etr"] = np.where(out["flag_low_etr"], np.maximum(_lowmag(z_etr), _lowmag(z_taxrev)), 0)

    # 2. Транзит / shell: высокая выручка на сотрудника + маленькая численность
    out["flag_shell"] = _false_na((z_rph >= thr) & (cy["headcount"].values <= config["shell_headcount_max"]))
    out["sev_shell"] = np.where(out["flag_shell"], _mag(z_rph), 0)

    # 3. Дивидендный вывод: высокий payout или дивиденды > чистой прибыли
    div_gt_net = _false_na((cy["dividends_paid"].values > cy["net_profit"].values) & (cy["dividends_paid"].values > 0))
    out["flag_dividend"] = _false_na(z_div >= thr) | div_gt_net
    out["sev_dividend"] = np.where(out["flag_dividend"], np.maximum(_mag(z_div), np.where(div_gt_net, thr, 0)), 0)

    # 4. Аномалия маржи относительно отрасли
    out["flag_margin"] = _false_na((np.abs(z_opm) >= thr) | (np.abs(z_gm) >= thr))
    out["sev_margin"] = np.where(out["flag_margin"], np.maximum(_mag(z_opm), _mag(z_gm)), 0)

    # ---- Tier 2 ----
    # 5. Нарушение бухгалтерских тождеств P&L
    out["flag_identity"] = _false_na(
        (cy["id_gross_resid"].values > config["identity_tol"]) | (cy["id_op_resid"].values > config["identity_tol"])
    )
    out["sev_identity"] = np.where(out["flag_identity"], thr, 0)

    # 6. Резкий скачок / обвал выручки YoY
    out["flag_yoy"] = _false_na(z_yoy >= config["yoy_flag_z"])
    out["sev_yoy"] = np.where(out["flag_yoy"], _mag(z_yoy), 0)

    # 7. Молодая компания с очень большой выручкой
    out["flag_young_big"] = _false_na((cy["company_age"].values <= config["young_age_max"]) & (z_rev >= thr))
    out["sev_young_big"] = np.where(out["flag_young_big"], _mag(z_rev), 0)

    # 8. Аномалия фонда оплаты труда на сотрудника
    out["flag_wage"] = _false_na(np.abs(z_pph) >= thr)
    out["sev_wage"] = np.where(out["flag_wage"], _mag(z_pph), 0)

    out["etr_bench_level"] = robust_z_by_group(cy["etr"].values, g)[1]
    return out


# Загрузка данных


In [ ]:
EXPECTED_COLS = [
    "company_id", "company_name", "inn", "ogrn_year", "region",
    "okved_code", "okved_section", "founder_id", "address_hash", "year",
    "revenue", "cost_of_goods", "gross_profit", "opex", "operating_profit",
    "net_profit", "taxes_paid", "headcount", "payroll_fund", "dividends_paid",
]

NUMERIC_COLS = [
    "ogrn_year", "year", "revenue", "cost_of_goods", "gross_profit", "opex",
    "operating_profit", "net_profit", "taxes_paid", "headcount",
    "payroll_fund", "dividends_paid",
]


def load_data(config=CONFIG):
    """Читает CSV, приводит типы, валидирует схему."""
    path = Path(config["data_path"])
    is_sample = False
    if not path.exists():
        print(f"WARNING: '{path}' не найден — используем выборку '{config['data_path_fallback']}'")
        path = Path(config["data_path_fallback"])
        is_sample = True
    if not path.exists():
        raise FileNotFoundError(f"Файл данных не найден: {path}")

    dat = pd.read_csv(path)
    missing = set(EXPECTED_COLS) - set(dat.columns)
    if missing:
        raise ValueError(f"В датасете отсутствуют колонки: {missing}")

    dat = dat[EXPECTED_COLS].copy()
    for col in NUMERIC_COLS:
        dat[col] = pd.to_numeric(dat[col], errors="coerce")

    dat.attrs["source_path"] = str(path)
    dat.attrs["is_sample"] = is_sample
    print(f"Загружено {len(dat)} строк x {len(dat.columns)} колонок из {path}" + (" [SAMPLE]" if is_sample else ""))
    return dat


# Генерация признаков


In [ ]:
def safe_ratio(num, den, den_positive_only=True):
    """Безопасное деление: при плохом знаменателе возвращает NA."""
    num = np.asarray(num, dtype=float)
    den = np.asarray(den, dtype=float)
    out = num / den
    if den_positive_only:
        bad = (den <= 0) | np.isnan(den)
    else:
        bad = (den == 0) | np.isnan(den)
    out[bad] = np.nan
    return out


def build_company_year_features(dat):
    """Признаки на уровне company-year: отношения и остатки тождеств."""
    cy = dat.copy()
    cy["company_age"] = cy["year"] - cy["ogrn_year"]
    cy["etr"] = safe_ratio(cy["taxes_paid"], cy["operating_profit"])
    cy["tax_to_revenue"] = safe_ratio(cy["taxes_paid"], cy["revenue"])
    cy["gross_margin"] = safe_ratio(cy["gross_profit"], cy["revenue"])
    cy["operating_margin"] = safe_ratio(cy["operating_profit"], cy["revenue"])
    cy["net_margin"] = safe_ratio(cy["net_profit"], cy["revenue"])
    cy["revenue_per_head"] = safe_ratio(cy["revenue"], cy["headcount"])
    cy["payroll_per_head"] = safe_ratio(cy["payroll_fund"], cy["headcount"])
    cy["dividend_payout"] = safe_ratio(cy["dividends_paid"], cy["net_profit"])
    cy["id_gross_resid"] = safe_ratio(
        np.abs((cy["revenue"] - cy["cost_of_goods"]) - cy["gross_profit"]),
        np.abs(cy["revenue"]),
        den_positive_only=False,
    )
    cy["id_op_resid"] = safe_ratio(
        np.abs((cy["gross_profit"] - cy["opex"]) - cy["operating_profit"]),
        np.abs(cy["revenue"]),
        den_positive_only=False,
    )
    return cy


def build_company_panel_features(cy):
    """Панельные признаки компании за 2023–2025: YoY, волатильность."""
    cy = cy.sort_values(["company_id", "year"]).copy()
    cy["revenue_yoy"] = cy.groupby("company_id")["revenue"].pct_change()
    panel = cy.groupby("company_id").agg(
        n_years=("year", "count"),
        max_abs_revenue_yoy=("revenue_yoy", lambda s: np.nanmax(np.abs(s))),
        revenue_cv=("revenue", lambda s: s.std() / s.mean() if s.mean() > 0 else np.nan),
    ).reset_index()
    panel["max_abs_revenue_yoy"] = panel["max_abs_revenue_yoy"].replace([np.inf, -np.inf], np.nan)
    return panel


# Скоринг


In [ ]:
def score_company_year(typ, config=CONFIG):
    """Композитный скор на уровне company-year: взвешенная сумма severity."""
    typ = typ.copy()
    w1, w2 = config["tier_weights"]["1"], config["tier_weights"]["2"]
    tier1_sev = sum(typ[FLAG_SEV_MAP[f]] for f in TIER1_FLAGS)
    tier2_sev = sum(typ[FLAG_SEV_MAP[f]] for f in TIER2_FLAGS)
    typ["cy_score"] = w1 * tier1_sev + w2 * tier2_sev
    typ["n_flags"] = sum(typ[f].astype(int) for f in TIER1_FLAGS + TIER2_FLAGS)
    typ["n_tier1"] = sum(typ[f].astype(int) for f in TIER1_FLAGS)
    return typ


def score_company(typ_scored, cy, config=CONFIG):
    """Агрегация до одной строки на компанию + материальность."""
    mat = cy.groupby("company_id").agg(
        materiality=("operating_profit", lambda s: np.nanmax(np.maximum(s, 0))),
        revenue_max=("revenue", "max"),
        taxes_paid_total=("taxes_paid", "sum"),
        revenue_cv=("revenue_cv", "first"),
    ).reset_index()

    flag_cols = TIER1_FLAGS + TIER2_FLAGS
    comp = typ_scored.groupby(["company_id", "company_name"]).agg(
        okved_section=(config["group_col"], "first"),
        composite=("cy_score", "sum"),
        n_flags=("n_flags", "sum"),
        n_tier1=("n_tier1", "sum"),
        **{f: (f, lambda s, _f=f: int(s.any())) for f in flag_cols},
    ).reset_index()
    return comp.merge(mat, on="company_id")


def bucket_findings(comp, config=CONFIG):
    """Корзины приоритета: urgent / background / economic_signal / not_flagged."""
    comp = comp.copy()
    flagged = comp["n_flags"] > 0
    lik_cut = comp.loc[flagged, "composite"].quantile(config["likelihood_hi_q"]) if flagged.any() else 0
    mat_cut = comp["materiality"].quantile(config["materiality_hi_q"])
    if not np.isfinite(lik_cut):
        lik_cut = 0
    if not np.isfinite(mat_cut):
        mat_cut = 0

    hi_lik = (comp["composite"] >= lik_cut) & flagged
    hi_mat = comp["materiality"] >= mat_cut
    consistent = comp["revenue_cv"].notna() & (comp["revenue_cv"] < 0.25)

    bucket = np.full(len(comp), "not_flagged", dtype=object)
    bucket[flagged & ~hi_lik] = "background"
    bucket[flagged & hi_lik & ~hi_mat] = "background"
    bucket[flagged & hi_lik & hi_mat] = "urgent"
    bucket[~flagged & hi_mat & consistent] = "economic_signal"

    comp["likelihood_high"] = hi_lik
    comp["materiality_high"] = hi_mat
    comp["bucket"] = pd.Categorical(
        bucket,
        categories=["urgent", "background", "economic_signal", "not_flagged"],
        ordered=True,
    )
    comp["_urgent_sort"] = (comp["bucket"] == "urgent").astype(int)
    return comp.sort_values(["_urgent_sort", "composite", "materiality"], ascending=[False, False, False]).drop(columns="_urgent_sort")


# Кластеры аффилированности

Ключ группы — `founder_id`; совпадение `address_hash` усиливает связь до `strong`.
Групповая типология: перераспределение прибыли — широкий разброс ETR / маржи внутри группы.


In [ ]:
def _shares_address(hashes):
    """Есть ли общий адрес у нескольких компаний в группе."""
    all_hashes = []
    for h in hashes:
        all_hashes.extend(h.split("|"))
    return any(v > 1 for v in Counter(all_hashes).values())


def build_groups(cy):
    """Группы связанных компаний по учредителю."""
    by_company = cy.groupby(["company_id", "founder_id"]).agg(
        address_hashes=("address_hash", lambda s: "|".join(sorted(s.unique()))),
        mean_etr=("etr", "mean"),
        mean_op_margin=("operating_margin", "mean"),
        revenue_max=("revenue", "max"),
        taxes_total=("taxes_paid", "sum"),
    ).reset_index()

    groups = by_company.groupby("founder_id").agg(
        n_companies=("company_id", "nunique"),
        companies=("company_id", lambda s: ", ".join(sorted(s.unique()))),
        shares_address=("address_hashes", _shares_address),
        etr_spread=("mean_etr", lambda s: np.nanmax(s) - np.nanmin(s)),
        margin_spread=("mean_op_margin", lambda s: np.nanmax(s) - np.nanmin(s)),
        group_revenue=("revenue_max", "sum"),
        group_taxes=("taxes_total", "sum"),
    ).reset_index()

    groups = groups[groups["n_companies"] >= 2].copy()
    groups["link_strength"] = np.where(groups["shares_address"], "strong", "founder_only")
    return groups.sort_values(["etr_spread", "group_revenue"], ascending=[False, False])

def build_affiliation_graph(cy, groups, comp=None):
    """Двудольный граф: учредители (круги) — компании (квадраты)."""
    if len(groups) == 0:
        return None

    edges = (
        cy[["company_id", "founder_id"]]
        .drop_duplicates()
        .query("founder_id in @groups['founder_id']")
    )
    if len(edges) == 0:
        return None

    ls_map = dict(zip(groups["founder_id"], groups["link_strength"]))
    bucket_map = dict(zip(comp["company_id"], comp["bucket"])) if comp is not None else {}

    return {
        "edges": [
            (row.founder_id, row.company_id, ls_map.get(row.founder_id))
            for row in edges.itertuples(index=False)
        ],
        "founders": sorted(edges["founder_id"].unique()),
        "bucket_map": bucket_map,
    }


def _layout_affiliation(graph):
    """Раскладка по кластерам: учредитель в центре, компании по окружности."""
    by_founder = defaultdict(list)
    for founder, company, _ in graph["edges"]:
        by_founder[founder].append(company)

    pos = {}
    x_offset = 0.0
    gap = 5.0
    for founder in graph["founders"]:
        companies = sorted(set(by_founder[founder]))
        cx, cy = x_offset, 0.0
        pos[founder] = (cx, cy)
        n = len(companies)
        if n:
            angles = np.linspace(0, 2 * np.pi, n, endpoint=False)
            r = 1.8 + 0.15 * n
            for company, angle in zip(companies, angles):
                pos[company] = (cx + r * np.cos(angle), cy + r * np.sin(angle))
        x_offset += gap
    return pos


def plot_affiliation_graph(graph, title="Группы аффилированности"):
    """Визуализация графа аффилированности (аналог R/05_groups.R)."""
    if graph is None or not graph["edges"]:
        return

    bucket_colors = {
        "urgent": "#D55E00",
        "background": "#0072B2",
        "economic_signal": "#E69F00",
        "not_flagged": "#999999",
    }

    pos = _layout_affiliation(graph)
    fig, ax = plt.subplots(figsize=(10, 7))

    for founder, company, link_strength in graph["edges"]:
        x0, y0 = pos[founder]
        x1, y1 = pos[company]
        color = "#222222" if link_strength == "strong" else "#666666"
        width = 5.0 if link_strength == "strong" else 3.0
        ax.plot([x0, x1], [y0, y1], color=color, linewidth=width, solid_capstyle="round", zorder=1)

    for founder in graph["founders"]:
        x, y = pos[founder]
        ax.scatter(x, y, s=420, c="#56B4E9", marker="o", zorder=2)
        ax.text(x, y + 0.25, founder, fontsize=8, ha="center", color="#222222")

    companies = {c for _, c, _ in graph["edges"]}
    for company in companies:
        x, y = pos[company]
        bucket = graph["bucket_map"].get(company, "not_flagged")
        color = bucket_colors.get(bucket, "#CCCCCC")
        ax.scatter(x, y, s=170, c=color, marker="s", zorder=2)
        if bucket in ("urgent", "background", "economic_signal"):
            ax.text(x, y, company, fontsize=6, ha="center", va="center", color="#222222")

    ax.set_title(title)
    ax.axis("off")

    legend_items = [
        ("Учредитель", "#56B4E9", "o"),
        ("Компания (urgent)", bucket_colors["urgent"], "s"),
        ("Компания (background)", bucket_colors["background"], "s"),
        ("Компания (economic signal)", bucket_colors["economic_signal"], "s"),
        ("Компания (без флагов)", bucket_colors["not_flagged"], "s"),
    ]
    handles = [
        plt.Line2D([0], [0], marker=m, color="w", markerfacecolor=c, markersize=9, label=lab)
        for lab, c, m in legend_items
    ]
    handles.append(plt.Line2D([0], [0], color="#222222", linewidth=4, label="Связь: общий адрес"))
    ax.legend(handles=handles, loc="lower left", frameon=False, fontsize=8)
    plt.tight_layout()
    plt.show()


# Генерация отчётов


In [ ]:
def run_pipeline(config=CONFIG):
    """Полный пайплайн: загрузка -> признаки -> типологии -> скоринг -> группы."""
    dat = load_data(config)
    cy = build_company_year_features(dat)
    panel = build_company_panel_features(cy)
    cy = cy.merge(panel, on="company_id", how="left")

    typ = run_typologies(cy, config)
    typ = score_company_year(typ, config)
    comp = score_company(typ, cy, config)
    comp = bucket_findings(comp, config)
    groups = build_groups(cy)

    return {
        "dat": dat,
        "cy": cy,
        "typ": typ,
        "comp": comp,
        "groups": groups,
        "is_sample": dat.attrs.get("is_sample", False),
    }


res = run_pipeline(CONFIG)
comp = res["comp"]
n_comp = comp["company_id"].nunique()
n_urg = (comp["bucket"] == "urgent").sum()
n_bg = (comp["bucket"] == "background").sum()
n_sig = (comp["bucket"] == "economic_signal").sum()

print(f"Компаний: {n_comp} | Срочных: {n_urg} | Фоновых: {n_bg} | Экон. сигналов: {n_sig}")


## Что нашли


In [ ]:
from IPython.display import Markdown, display

sample_note = ' *(демо-выборка)*' if res['is_sample'] else ''
display(Markdown(f'''Из **{n_comp}** компаний выявлено **{n_urg}** приоритетных (высокий риск × высокая
материальность), **{n_bg}** фоновых и **{n_sig}** случаев, которые являются реальными
экономическими сигналами, а не аномалиями{sample_note}.

Анализ ведётся через призму **налогового риска** (ADR-001): восемь типологий, из которых четыре — ведущие
(низкая эффективная ставка налога, «технические»/транзитные фирмы, вывод прибыли через дивиденды,
аномалии маржи относительно отрасли).
'''))


## Как проверяли


In [ ]:
display(Markdown(f'''
Отклонения измеряются **относительно отрасли** (`okved_section`) робастной статистикой
(медиана/MAD, порог |z| ≥ {CONFIG['robust_z_flag']}) с откатом на глобальную норму для малых
отраслей (ADR-004). Арифметические невозможности (дивиденды > чистой прибыли, нарушение тождеств
P&L) проверяются абсолютными правилами. Приоритет = **вероятность × материальность** (ADR-005).
'''))


## Приоритетная карта


In [ ]:
plot_dat = comp[comp['n_flags'] > 0]
if len(plot_dat) > 0:
    colors = {'urgent': 'C3', 'background': 'C0', 'economic_signal': 'C2', 'not_flagged': 'C7'}
    for bucket, grp in plot_dat.groupby('bucket', observed=True):
        plt.scatter(grp['composite'], grp['materiality'], alpha=0.75, s=30,
                    label=bucket, c=colors.get(bucket, 'C7'))
    plt.xlabel('Композитный риск-скор (вероятность)')
    plt.ylabel('Материальность (макс. операц. прибыль, ₽)')
    plt.legend(title='Корзина')
    plt.tight_layout()
    plt.show()
else:
    print('Нет помеченных компаний в выборке.')


## Частота срабатывания типологий


In [ ]:
flag_cols = TIER1_FLAGS + TIER2_FLAGS
freq = pd.DataFrame({
    'typology': flag_cols,
    'n': [comp[f].sum() for f in flag_cols],
    'tier': ['Tier 1' if f in TIER1_FLAGS else 'Tier 2' for f in flag_cols],
})
colors = {'Tier 1': 'C0', 'Tier 2': 'C1'}
freq = freq.sort_values('n')
plt.barh(freq['typology'], freq['n'], color=[colors[t] for t in freq['tier']])
plt.xlabel('Компаний')
plt.tight_layout()
plt.show()


## Топ приоритетных находок


In [ ]:
top_urgent = (
    comp[comp["bucket"] == "urgent"]
    .sort_values("composite", ascending=False)
    .head(10)
    .assign(
        composite=lambda d: d["composite"].round(1),
        materiality_fmt=lambda d: d["materiality"].round(0).map(lambda x: f"{x:,.0f}"),
    )[
        ["company_id", "company_name", "okved_section", "composite", "n_flags", "materiality_fmt"]
    ]
    .rename(columns={"n_flags": "флагов", "materiality_fmt": "материальность, руб."})
)
display(top_urgent)


## Связанные группы (аффилированность)


### Граф аффилированности

Двудольный граф: **круги** — учредители (`founder_id`), **квадраты** — компании.
Цвет квадрата — корзина приоритета; толстые тёмные рёбра — группы с общим адресом (*strong*).


In [ ]:
aff_graph = build_affiliation_graph(res['cy'], res['groups'], comp)
if aff_graph is not None:
    plot_affiliation_graph(aff_graph, title='Группы аффилированности')
else:
    print('Граф аффилированности не построен: нет групп с >=2 компаниями.')


In [ ]:
groups = res["groups"]
if len(groups) > 0:
    display(
        groups.head(8).assign(
            разброс_ETR=lambda d: d["etr_spread"].round(3),
        )[
            ["founder_id", "n_companies", "link_strength", "разброс_ETR", "companies"]
        ].rename(columns={
            "founder_id": "владелец",
            "n_companies": "кол-во компаний",
            "link_strength": "сила связи",
            "companies": "компании",
        })
    )
else:
    print("Групп из >=2 компаний с общим учредителем не обнаружено.")
